# Fundamentals 13 - Multi-Agentic Graph End-to-End

Objetivo: tomar los mismos roles del notebook 12 y cambiar solamente la orquestacion: de una secuencia explicita a un Graph con estado, nodos y edges.

```text
GraphState
   |
START -> solve -> judge -> review -> END
                    |
                    +-> optional LM degradation
```

El System sigue siendo owner de los Agents. El Graph es owner del flujo y del estado compartido. Esa separacion evita confundir composicion de Agents con orquestacion.


In [ ]:
import importlib.util
from typing import TypedDict
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=8, max_turns=8)
local_runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.system(runtime=local_runtime)
toolkit.show({
    "local_runtime": local_runtime.describe(),
    "lm_runtime": lm_resolution,
    "lm_available": lm_available,
    "force_local_only": force_local_only,
}, title="Runtime selection")


## Escenario didactico compartido

Se conservan solver, judge, reviewer, prompt y resultado esperado del notebook 12. Repetir el dominio hace visible que el cambio esta en la topologia de ejecucion.


## Parámetros de la integración multi-agent Graph

| Parametro | Qué controla | Decision del notebook |
|---|---|---|
| `GraphState` | Datos y resultados compartidos entre nodos. | Conserva procedure, result, judge y `RunResult` internos. |
| `nodes` | Funciones ejecutables del Graph. | `solve`, `judge` y `review`. |
| `edges` | Orden y limites del flujo. | `START -> solve -> judge -> review -> END`. |
| `engine="langgraph"` | Adaptador externo de orquestacion. | Se usa cuando LangGraph esta instalado. |
| fallback local | Ejecucion sin LangGraph. | Recorre los mismos nodos en el mismo orden. |
| `local_runtime` | Runtime de solve y judge. | `python-runtime`. |
| `lm_runtime` | Runtime del nodo review opcional. | `provider="auto"` para OpenAI, Bedrock o vLLM. |
| `AGENTIC_SYSTEMS_PROVIDER_PRIORITY` | Orden de Providers. | Cambia backend sin modificar la topologia. |


## 1) Declarar Agents y estado compartido

Los Agents siguen registrados en un solo `AgenticSystem`. `GraphState` agrega los resultados internos necesarios para que la composicion final conserve Tools y lineage.


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revision LM como evidencia estructurada."""
    return {"summary": summary}

USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42

@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}

@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}

policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solver = system.agent(name="graph_solver", instructions="Resuelve numeros estructurados.", tools=[solve_arithmetic], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic")), policy=policy)
judge = system.agent(name="graph_judge", instructions="Valida resultado.", tools=[judge_result], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["judge_result"], tool_expectation=toolkit.expect.exactly("judge_result")), policy=policy)
reviewer = system.agent(name="graph_lm_reviewer", instructions="Resume riesgos sin cambiar el resultado.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))

class GraphState(TypedDict, total=False):
    prompt: str
    numbers: list[int]
    procedure: list[str]
    result: float
    judge: dict
    lm_review: str | None
    solve_result: object
    judge_result: object
    review_result: object | None
    review_diagnostic: dict

toolkit.show({
    "prompt": USER_PROMPT,
    "numbers": NUMBERS,
    "expected": EXPECTED,
    "system": system.inspect(),
    "agents": [solver.info(), judge.info(), reviewer.info()],
}, title="Multi-agent graph inputs and ownership")


## 2) Convertir responsabilidades en nodos

Cada nodo recibe y devuelve estado. El nodo review puede degradarse y dejar un diagnostico sin borrar el resultado producido por solve y validado por judge.


In [ ]:
def solve_node(state: GraphState) -> GraphState:
    result = solver.run({"tool": "solve_arithmetic", "input": {"numbers": state["numbers"]}})
    return {**state, "procedure": result.data["procedure"], "result": result.data["result"], "solve_result": result}

def judge_node(state: GraphState) -> GraphState:
    result = judge.run({"tool": "judge_result", "input": {"result": state["result"], "expected": EXPECTED}})
    return {**state, "judge": result.data, "judge_result": result}

def review_node(state: GraphState) -> GraphState:
    if not lm_available:
        diagnostic = {
            "status": "skipped",
            "provider": lm_resolution.get("selected_provider"),
            "reason": lm_resolution.get("reason"),
        }
        return {**state, "lm_review": None, "review_result": None, "review_diagnostic": diagnostic}

    attempt = reviewer.run(str({"procedure": state["procedure"], "result": state["result"], "judge": state["judge"]}))
    if attempt.ok:
        diagnostic = {"status": "ok", "provider": attempt.engine, "model": attempt.model}
        return {**state, "lm_review": attempt.text, "review_result": attempt, "review_diagnostic": diagnostic}

    diagnostic = {"status": "degraded", "provider": attempt.engine, "errors": attempt.errors}
    toolkit.show(diagnostic, title="Optional LM review degraded")
    return {**state, "lm_review": None, "review_result": None, "review_diagnostic": diagnostic}

nodes = {"solve": solve_node, "judge": judge_node, "review": review_node}
edges = [("START", "solve"), ("solve", "judge"), ("judge", "review"), ("review", "END")]
toolkit.show({"nodes": list(nodes), "edges": edges})

## 3) Ejecutar la misma topologia

Si LangGraph esta disponible, `toolkit.graph` ejecuta la topologia. El fallback local conserva los mismos nodos y edges para que el tutorial siga siendo ejecutable.


In [ ]:
initial_state: GraphState = {"prompt": USER_PROMPT, "numbers": NUMBERS}
if importlib.util.find_spec("langgraph"):
    graph = toolkit.graph(state=GraphState, nodes=nodes, edges=edges, engine="langgraph", name="fundamentals_multi_agentic_graph")
    final_state = graph.run(initial_state)
    framework = "langgraph"
else:
    final_state = initial_state
    for node in [solve_node, judge_node, review_node]:
        final_state = node(final_state)
    framework = "local-state-pipeline"


## 4) Componer resultado y lineage

Los `RunResult` guardados en el estado se consolidan al final. De este modo el Graph no pierde Tools, engines ni diagnosticos al proyectar su estado a una respuesta humana.


In [ ]:
payload = {
    "procedimiento": final_state["procedure"],
    "resultado_final": final_state["result"],
    "judge": final_state["judge"],
    "lm_review": final_state.get("lm_review"),
}
result = toolkit.compose_result(
    text="Graph multi-agente ejecutado.",
    data=payload,
    results=[final_state.get("solve_result"), final_state.get("judge_result"), final_state.get("review_result")],
    mode="multi-agentic-graph",
    framework=framework,
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution, "optional_lm_review": final_state.get("review_diagnostic")},
)
lineage = result.lineage(name="fundamentals.multi_agentic_graph", question=USER_PROMPT, goal="Explicar nodos, edges y estado final.")
toolkit.show({"framework": framework, "final_state_keys": list(final_state)})
toolkit.human_result(result, title="Human result - Multi-Agentic Graph", pretty=PRETTY, show_lineage=True, lineage=lineage)

## Lo que aprendiste

- `AgenticSystem` posee Agents; Graph posee estado y orden de ejecucion.
- Los nodos deben propagar los `RunResult` necesarios para conservar evidencia.
- LangGraph y el fallback local pueden compartir la misma topologia declarada.
- Un nodo LM opcional puede degradarse sin invalidar los nodos obligatorios.
- Graph no reemplaza la composicion del notebook 12: la hace explicita como estructura.
- Environment y Eval no se repiten aqui; sus fundamentos pertenecen al notebook 10.


## Coverage API de este notebook

La cobertura se concentra en la frontera entre System, Agents y Graph, ademas de la preservacion del contrato central.


In [ ]:
api_coverage = [
    "AgenticSystem",
    "AgenticSystem.agent",
    "toolkit.graph",
    "Graph state",
    "nodes",
    "edges",
    "runtime(provider='auto')",
    "compose_result",
    "RunResult.lineage",
]
toolkit.show({
    "notebook": "13_multi_agentic_graph_api.ipynb",
    "api_coverage": api_coverage,
})

## Simbolos API explicados

- `AgenticSystem`: owner de solver, judge y reviewer.
- `toolkit.graph`: construye el Graph nativo con adaptador LangGraph cuando esta disponible.
- `GraphState`: contrato local del estado compartido.
- `nodes` / `edges`: topologia ejecutable e inspeccionable.
- `compose_result`: proyecta resultados de nodos a un `RunResult` auditable.
- `RunResult.lineage`: explica la ruta y la evidencia preservada por el Graph.
